<a href="https://colab.research.google.com/github/Jerrylu99/5200-Project/blob/main/lab13/lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('Zillow_California_2026_Hedonic.csv')
df.head()


,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [ ]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()

In [ ]:
print(naive_model.summary())

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:32:27   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [ ]:
print("Naive Age Coefficient:", naive_model.params['Property_Age'])

Naive Age Coefficient: 5573.630595439929


In [ ]:
if naive_model.params['Property_Age'] < 0:
    print("The coefficient for 'Property_Age' is negative, indicating that in the naive model, the older the property, the lower the price")
else:
    print("The coefficient for 'Property_Age' is positive, indicating that in the naive model, the older the property, the higher the price。")

The coefficient for 'Property_Age' is positive, indicating that in the naive model, the older the property, the higher the price。


In [ ]:
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

In [ ]:
print(multi_model.summary())

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:32:27   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [ ]:
print("Multivariate Age Coefficient:", multi_model.params['Property_Age'])

Multivariate Age Coefficient: -2063.1292168021246


In [ ]:
print("Step 1 coefficient:", naive_model.params['Property_Age'])
print("Step 2 coefficient:", multi_model.params['Property_Age'])

Step 1 coefficient: 5573.630595439929
Step 2 coefficient: -2063.1292168021246


In [ ]:
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

In [ ]:
df[['Sale_Price', 'Distance_to_Tech_Hub', 'Price_Residuals']].head()

,Sale_Price,Distance_to_Tech_Hub,Price_Residuals
0,684100.56,38.1,-56823.332740
1,413634.22,95.1,19000.990249
2,456709.35,73.5,-69149.815200
3,624533.95,60.3,18481.157582
4,870137.54,16.4,-2619.815668


In [ ]:
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [ ]:
df[['Property_Age', 'Distance_to_Tech_Hub', 'Age_Residuals']].head()

,Property_Age,Distance_to_Tech_Hub,Age_Residuals
0,77.5,38.1,0.621803
1,11.0,95.1,-13.689856
2,47.7,73.5,3.233510
3,61.9,60.3,5.347789
4,100.8,16.4,4.053610


In [ ]:
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()

In [ ]:
print(fwl_model.summary())

                                 OLS Regression Results                                
Dep. Variable:        Price_Residuals   R-squared (uncentered):                   0.217
Model:                            OLS   Adj. R-squared (uncentered):              0.216
Method:                 Least Squares   F-statistic:                              276.5
Date:                Fri, 13 Mar 2026   Prob (F-statistic):                    5.27e-55
Time:                        18:32:27   Log-Likelihood:                         -11982.
No. Observations:                1000   AIC:                                  2.397e+04
Df Residuals:                     999   BIC:                                  2.397e+04
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------

In [ ]:
print("FWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

FWL Isolated Age Coefficient: -2063.129216802139


In [ ]:
print("Step 2 Property_Age coefficient:", multi_model.params['Property_Age'])
print("Step 3 FWL coefficient:", fwl_model.params['Age_Residuals'])

Step 2 Property_Age coefficient: -2063.1292168021246
Step 3 FWL coefficient: -2063.129216802139


In [ ]:
print("Difference:", multi_model.params['Property_Age'] - fwl_model.params['Age_Residuals'])

Difference: 1.4551915228366852e-11


In [ ]:
print("Step 2 rounded:", round(multi_model.params['Property_Age'], 6))
print("Step 3 rounded:", round(fwl_model.params['Age_Residuals'], 6))

Step 2 rounded: -2063.129217
Step 3 rounded: -2063.129217


In [ ]:
import plotly.graph_objects as go
import numpy as np

# extract coefficients from multivariate regression
b0 = multi_model.params['Intercept']
b1 = multi_model.params['Property_Age']
b2 = multi_model.params['Distance_to_Tech_Hub']

# create grid for visualization
age_range = np.linspace(df.Property_Age.min(), df.Property_Age.max(), 40)
dist_range = np.linspace(df.Distance_to_Tech_Hub.min(), df.Distance_to_Tech_Hub.max(), 40)

age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# predicted surface
price_pred = b0 + b1 * age_grid + b2 * dist_grid

# create 3D plot
fig = go.Figure()

# scatter data
fig.add_trace(
    go.Scatter3d(
        x=df.Property_Age,
        y=df.Distance_to_Tech_Hub,
        z=df.Sale_Price,
        mode='markers',
        marker=dict(size=4),
        name='Observed Data'
    )
)

# regression plane
fig.add_trace(
    go.Surface(
        x=age_grid,
        y=dist_grid,
        z=price_pred,
        opacity=0.5,
        name='Regression Plane'
    )
)

fig.update_layout(
    title="3D Visualization of Hedonic Pricing Regression",
    scene=dict(
        xaxis_title="Property Age",
        yaxis_title="Distance to Tech Hub",
        zaxis_title="Sale Price"
    )
)

fig.show()

### Interpretation

The visualization shows the regression hyperplane estimated from the multivariate model.
Each point represents an observed property in the dataset, while the surface represents the predicted housing price conditional on Property Age and Distance to the Tech Hub.

The slope of the plane illustrates how the partial effect of Property Age changes while holding Distance_to_Tech_Hub constant.

